# Section 05 lab — Joins and Parquet output

Each exercise creates its own source DataFrames.


## Exercise 1 — Compare inner, left and left-anti joins


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())

transactions = spark.createDataFrame([('T1', 'P1'), ('T2', 'P2'), ('T3', 'P999')], ['transaction_id', 'product_id'])
products = spark.createDataFrame([('P1', 'Mouse'), ('P2', 'Keyboard')], ['product_id', 'product_name'])
print('Inner join')
transactions.join(products, 'product_id', 'inner').show()
print('Left join')
transactions.join(products, 'product_id', 'left').show()
print('Unknown product IDs')
unknown = transactions.join(products, 'product_id', 'left_anti')
unknown.show()
assert unknown.first()['product_id'] == 'P999'


## Exercise 2 — Check reference-key uniqueness


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

product_schema = StructType([
    StructField('product_id', StringType()), StructField('product_name', StringType()),
    StructField('category', StringType()), StructField('list_price', DoubleType())])
products = spark.read.schema(product_schema).json('../data/products.json')
duplicate_keys = products.groupBy('product_id').count().filter(F.col('count') > 1)
duplicate_keys.show()
assert duplicate_keys.count() == 0
print('All product keys are unique.')


## Exercise 3 — Write and read partitioned Parquet


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F

rows = [('O1', '2026-10-01', 'Accessories', 1499.0),
        ('O2', '2026-10-02', 'Audio', 2499.0),
        ('O3', '2026-11-01', 'Electronics', 12999.0)]
sales = (spark.createDataFrame(rows, ['order_id', 'order_date_text', 'category', 'revenue'])
         .withColumn('order_date', F.to_date('order_date_text'))
         .withColumn('order_month', F.date_format('order_date', 'yyyy-MM'))
         .drop('order_date_text'))
output_path = '../output/section05_sales'
sales.write.mode('overwrite').partitionBy('order_month').parquet(output_path)
saved = spark.read.parquet(output_path)
saved.printSchema()
saved.orderBy('order_id').show()
assert saved.count() == 3
